# GLM-5.3 (753B) → Chronos v7.5 on a single Colab GPU

Two independent workflows:

1. **Convert** `zai-org/GLM-5.3` (755.6 GB block-FP8) into Chronos v7.5 holographic shards and stream them to a Hugging Face repo (resumable; the local disk only holds a few shards at a time).
2. **Run** the converted checkpoint with the standalone engine (`chronos_1t/models/glm53_engine.py`). vLLM's DSA sparse-MLA backends do not support sm_80, so A100 uses this engine; the DGX Spark uses vLLM (`scripts/start_glm53_vllm.sh`).

Recommended runtime: **A100 80GB (High-RAM)**, H100 80GB, or G4 (RTX PRO 6000, 96 GB).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2; df -h /content | tail -1
!git clone -q https://github.com/defnotean/Chronos.git /content/Chronos || (cd /content/Chronos && git pull -q)
%cd /content/Chronos
!pip install -q -e . "transformers>=5.15" safetensors huggingface_hub hf_xet
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e:
    print('Set HF_TOKEN manually if you want to upload converted shards:', e)
os.environ['TRITON_DEFAULT_FP_FUSION'] = '1'

## Footprint & residency plan
Shows the Chronos checkpoint sizes and how many experts per layer each device keeps in T1 (4.5 bpw) / T2 (2.25 bpw) / T3 (1.125 bpw), on the GPU vs. pinned host RAM.

In [ ]:
!python -m chronos_1t.models.glm53_chronos_spec --ctx 32768

## 1. Convert (optional — skip if the converted repo already exists)
~1.5–3 h on an A100 (download-bound). Re-run the cell after a disconnect; finished layers are skipped (it checks both the local dir and the upload repo). Use `--compact` for 1-bit tails (~199 GB instead of ~302 GB) and `--hot-experts 32` for an Upgrade-2 sidecar (+51.6 GB) with the 32 highest-prior experts/layer packed straight from FP8 into T1.

In [ ]:
UPLOAD_REPO = 'defnoteatin/PseudoBrain-GLM-5.3-Chronos-v7.5'
!python chronos_1t/converter/convert_glm53_chronos.py \
    --out-dir /content/glm53_chronos_v75 --cache-dir /content/hf_cache_glm53 \
    --upload-repo {UPLOAD_REPO} --delete-after-upload

## 2. Run on this GPU
Streams each converted layer shard from the Hub, builds its tier slots, then deletes the shard (`--stream-delete`). Set `CHRONOS_GLM53_TIER_CACHE` to keep the built tiers on disk for faster restarts (needs ~the resident size in free disk).

In [ ]:
!python scripts/run_glm53_chronos_colab.py --repo defnoteatin/PseudoBrain-GLM-5.3-Chronos-v7.5 \
    --model-dir /content/glm53_chronos --ctx 32768 --stream-delete --benchmark --max-new-tokens 256

## 3. OpenAI-compatible server (background)

In [ ]:
import subprocess, time
srv = subprocess.Popen(['python', 'scripts/run_glm53_chronos_colab.py', '--repo', 'defnoteatin/PseudoBrain-GLM-5.3-Chronos-v7.5',
                        '--model-dir', '/content/glm53_chronos', '--ctx', '32768', '--serve', '--port', '8000'],
                       stdout=open('/content/glm53_server.log', 'w'), stderr=subprocess.STDOUT)
print('started; tail -f /content/glm53_server.log until "[serve]" appears')

In [ ]:
!curl -s http://127.0.0.1:8000/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model":"glm-5.3-chronos","messages":[{"role":"user","content":"Write a binary search in Rust."}],"max_tokens":200}'